# Stress-test toàn pipeline

**Mục tiêu:** kiểm tra sự ổn định của pipeline — *phân cụm*, *ước lượng tác động*, *khuyến nghị*

| Trục | Cách lay | Tiêu chí ổn định |
|---|---|---|
| **1. Segmentation** | đổi seed KMeans · bootstrap rider | nhóm target còn tồn tại và giữ được thành viên |
| **2. Experiment** | đổi estimator · bỏ từng khối · hoán vị Fisher · bootstrap phân tầng | uplift còn dương, có ý nghĩa, vượt ngưỡng hoà vốn |
| **3. Khuyến nghị** | quét voucher × take rate · giải điểm lật từng giả định | quyết định triển khai không lật |



#### Mục lục

| Phần | Nội dung |
|---|---|
| **1. Stress segmentation** | |
| 1.1 Đổi random seed | 10 seed · ARI · Jaccard · precision–recall · truy target bằng centroid |
| 1.2 Bootstrap rider | 40 vòng · xác suất mỗi rider thuộc target · % rider lung lay |
| **2. Stress experiment** | |
| 2.1  | ba estimator (thô · block FE · + hiệp biến) · **bỏ từng khối** |
| 2.2 | hoán vị Fisher 10.000 lần · bootstrap phân tầng 2.000 lần |
| **3. Stress khuyến nghị** | điểm lật từng giả định · lưới voucher × take rate · vùng vận hành |

In [38]:
# Nạp thư viện
import os, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Khai báo đường dẫn
ROOT = r"c:/Users/Linh/Desktop/Growth & Experimentation Project for Ride-Hailing Promotions"
OUT  = os.path.join(ROOT, "05. Stress test", "outputs"); os.makedirs(OUT, exist_ok=True)

# Bảng màu dùng chung cho mọi biểu đồ
SURFACE, INK, SECOND, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SERIES, ALT     = "#e1e0d9", "#c3c2b7", "#2a78d6", "#eb6834"

SEED, ALPHA      = 42, 0.05
# Bốn đặc trưng hành vi dùng để phân cụm ở notebook 03
CLUSTER_FEATURES = ["pct_tip_rate", "pct_flex_payment", "pct_airport", "weekend_ratio"]
# Ba giả định kinh doanh, quyết định ngưỡng hoà vốn — phải khớp notebook 04
VOUCHER_VALUE, OPEX_PER_RIDER, TAKE_RATE = 5.00, 1.25, 0.20
COST_PER_RIDER = VOUCHER_VALUE + OPEX_PER_RIDER   # chi phí thật trên mỗi rider được gửi

# Đọc dữ liệu thí nghiệm và nhãn phân cụm rồi ghép theo user_id
exp = pd.read_csv(os.path.join(ROOT, "02. Synthetic_data", "output",
                               "experiment_ab_final.csv"))
seg = pd.read_csv(os.path.join(ROOT, "03. Segmentation", "outputs", "rider_cluster.csv"))

df = exp.merge(seg[["user_id", "cluster", "persona", "is_target"]],
               on="user_id", validate="1:1")
df["tau_true"] = df.ite_realized          # tác động thật của từng rider, có vì dữ liệu synthetic

# Chuẩn hoá bốn đặc trưng: KMeans dùng khoảng cách Euclid nên nếu không chuẩn hoá,
# đặc trưng có phương sai lớn nhất sẽ lấn át phép chia cụm.
X          = StandardScaler().fit_transform(df[CLUSTER_FEATURES].to_numpy(float))
BEST_K     = df.cluster.nunique()                            # số cụm notebook 03 đã chọn
TARGET     = int(df.loc[df.is_target == 1, "cluster"].iloc[0])   # id cụm target gốc
TARGET_SET = set(df.loc[df.cluster == TARGET, "user_id"])        # tập rider của cụm target
# Tâm của cụm target — chữ ký hành vi dùng để truy lại nhóm sau mỗi lần phân cụm lại
TARGET_CENTROID = X[df.cluster.values == TARGET].mean(0)


def ols_hc1(Xd, y, j=1):
    """Hệ số OLS và sai số chuẩn HC1 cho biến thứ j.

    Bản sao nguyên văn của hàm trong notebook 04 — phải giống hệt thì stress test
    mới đang lay đúng con số mà báo cáo đang dùng.
    """
    XtX   = Xd.T @ Xd
    beta  = np.linalg.solve(XtX, Xd.T @ y)
    r     = y - Xd @ beta
    nn, k = Xd.shape
    meat  = (Xd * (r ** 2)[:, None]).T @ Xd         # phần giữa của công thức sandwich
    A     = np.linalg.solve(XtX, meat)
    V     = np.linalg.solve(XtX, A.T).T * nn / (nn - k)
    return beta[j], np.sqrt(V[j, j])


def fe_ate_se(y, t, blk, extra=None):
    """ATE và sai số chuẩn HC1 của mô hình Y ~ T + block FE, có thể thêm hiệp biến."""
    if len(np.unique(t)) < 2:               # nửa mẫu có thể rơi vào trường hợp chỉ còn một nhánh
        return np.nan, np.nan
    parts = [np.ones(len(y)), t]
    if extra is not None:
        parts.append(extra)
    parts.append(pd.get_dummies(pd.Series(blk), drop_first=True).to_numpy(float))
    try:
        return ols_hc1(np.column_stack(parts), y)
    except np.linalg.LinAlgError:           # ma trận suy biến khi một khối mất hết quan sát
        return np.nan, np.nan


def fe_ate(y, t, blk):
    """Chỉ trả về điểm ước lượng, không tính sai số chuẩn.

    Dùng thủ thuật Frisch-Waugh: khử trung bình khối khỏi cả Y và T rồi hồi quy đơn
    biến. Cho đúng con số như fe_ate_se nhưng không phải dựng ma trận biến giả, nên
    đủ nhanh để gọi hàng chục nghìn lần trong vòng lặp bootstrap và hoán vị.
    """
    yb = y - pd.Series(y).groupby(blk).transform("mean").to_numpy()
    tb = t - pd.Series(t).groupby(blk).transform("mean").to_numpy()
    d  = tb @ tb
    return np.nan if d < 1e-9 else (tb @ yb) / d


def strata(*keys):
    """Danh sách chỉ số của từng ô phân tầng.

    Ghép nhiều khoá thành một mã phẳng rồi nhóm lại. Bootstrap bốc lại TRONG từng ô
    này để mẫu mô phỏng giữ đúng cấu trúc thiết kế của thí nghiệm.
    """
    codes = pd.MultiIndex.from_arrays(keys).codes
    flat  = np.zeros(len(keys[0]), np.int64)
    for c in codes:
        flat = flat * (c.max() + 1) + c
    return [np.flatnonzero(flat == v) for v in np.unique(flat)]


# Những con số của notebook 04 mà stress test sẽ đem ra lay
tg   = df[df.is_target == 1].copy()
Y_TG = tg.Y_rct.to_numpy(float)          # số chuyến
T_TG = tg.T_rct.to_numpy(float)          # nhận voucher hay không
B_TG = tg.block_id.to_numpy()            # khối bốc thăm

ATE, SE    = fe_ate_se(Y_TG, T_TG, B_TG)
Z          = stats.norm.ppf(1 - ALPHA / 2)
LO_A, HI_A = ATE - Z * SE, ATE + Z * SE                       # khoảng tin cậy giải tích
BREAKEVEN  = COST_PER_RIDER / (tg.avg_fare.mean() * TAKE_RATE)  # ngưỡng hoà vốn

print(f"{len(df):,} rider | K = {BEST_K} | cum target = {TARGET} ({len(TARGET_SET):,} rider)")
print(f"Truc phan cum : {', '.join(CLUSTER_FEATURES)}")
print(f"{len(tg):,} rider trong nhom target | {tg.block_id.nunique()} khoi")
print(f"\nKet qua notebook 04 dang bi lay:")
print(f"  ATE (block FE, HC1) : {ATE:.4f}   SE {SE:.4f}   KTC 95% [{LO_A:.4f}, {HI_A:.4f}]")
print(f"  BREAKEVEN           : {BREAKEVEN:.4f} chuyen / rider")

20,000 rider | K = 3 | cum target = 2 (6,446 rider)
Truc phan cum : pct_tip_rate, pct_flex_payment, pct_airport, weekend_ratio
6,446 rider trong nhom target | 10 khoi

Ket qua notebook 04 dang bi lay:
  ATE (block FE, HC1) : 1.9670   SE 0.1412   KTC 95% [1.6902, 2.2438]
  BREAKEVEN           : 1.3783 chuyen / rider


## 1. Stress segmentation
Mục tiêu: Kiểm tra sự ổn định của segmentation pipeline
| Thước đo | Giải thích |
|---|---|
| **ARI** | toàn bộ cách chia có giống cách chia gốc không? |
| **Jaccard** | tập rider target mới trùng tập gốc bao nhiêu? |
| **precision / recall** | nếu lệch thì lệch kiểu gì — nhóm bị *chẻ nhỏ* (precision cao, recall thấp) hay bị *trộn* với nhóm khác (precision thấp, recall cao)? |

#### 1.1. Nếu đổi random seed, thì nhóm target có còn giống nhóm target ban đầu không?

In [39]:
# Tìm lại target cluster sau khi chạy lại KMeans
def match_target_cluster(labels):
    """
    Sau khi chạy lại KMeans, số thứ tự cluster có thể bị đổi.
    Ví dụ:
        Lần chạy gốc:    Cluster 2 = Target
        Lần chạy mới:    Cluster 0 có thể chính là Target
    
    Vì vậy không thể cố định "Target = Cluster 2" nên cần 
    tìm cluster mới có centroid gần TARGET_CENTROID nhất.
    """
    # Lấy danh sách các cluster hiện tại
    ks  = np.unique(labels)
    # Tính centroid của từng cluster mới
    cen = np.array([X[labels == k].mean(0) for k in ks])
    # Tính khoảng cách bình phương giữa centroid cluster mới và centroid target gốc
    return int(ks[np.argmin(((cen - TARGET_CENTROID) ** 2).sum(1))])

# SO SÁNH TARGET MỚI VỚI TARGET GỐC
def overlap(new_ids):
    """
    So sánh danh sách rider trong target mới với target gốc.
    Trả về:
        Jaccard  : mức độ giống nhau tổng thể
        Precision: target mới có bao nhiêu % là target gốc
        Recall   : target gốc giữ lại được bao nhiêu %
    """

    # a = Target mới
    # b = Target gốc
    a, b = set(new_ids), TARGET_SET
    # Số rider xuất hiện ở cả Target mới và Target gốc
    inter = len(a & b)

    # Jaccard 
    jaccard = inter / len(a | b)

    # Precision = đúng / số được chọn mới
    # => Target mới có chọn nhầm nhiều không?
    precision = inter / len(a)

    # Recall = đúng / số Target gốc
    # => Target gốc có bị bỏ sót nhiều không?
    recall = inter / len(b)

    return jaccard, precision, recall

rows = []
# Cluster của segmentation gốc
base = df.cluster.values
# Id của rider 
uid  = df.user_id.values

# Stress test: Đổi seed
for s in range(SEED, SEED + 10):    
    # Chạy lại KMeans với BEST_K cố định, chỉ thay đổi random seed                  
    lab = KMeans(BEST_K, n_init=10, random_state=s).fit_predict(X)
    # Tìm lại Target cluster bằng centroid vì số thứ tự cluster có thể đã bị đổi
    k   = match_target_cluster(lab)
    # So sánh Target mới với Target gốc
    j, p, r = overlap(uid[lab == k])
    rows.append({"phep lay": "doi seed", "tham so": f"seed={s}",
                 "ARI vs goc": adjusted_rand_score(base, lab),
                 "Jaccard": j, "precision": p, "recall": r,
                 "n target": int((lab == k).sum())})
    

# Kết quả 
seg_stab = pd.DataFrame(rows)
display(seg_stab.round(4))

_sd = seg_stab[seg_stab["phep lay"] == "doi seed"]
_k1 = seg_stab[seg_stab["tham so"].isin([f"K={BEST_K - 1}", f"K={BEST_K + 1}"])]
print(f"Doi seed : ARI {_sd['ARI vs goc'].min():.4f}-{_sd['ARI vs goc'].max():.4f} | "
      f"Jaccard {_sd.Jaccard.min():.4f}-{_sd.Jaccard.max():.4f}")

,phep lay,tham so,ARI vs goc,Jaccard,precision,recall,n target
0,doi seed,seed=42,1.0000,1.0000,1.0000,1.0000,6446
1,doi seed,seed=43,0.9775,0.9786,0.9937,0.9846,6387
2,doi seed,seed=44,0.9943,0.9946,0.9949,0.9997,6477
3,doi seed,seed=45,0.9932,0.9937,0.9940,0.9997,6483
4,doi seed,seed=46,0.9884,0.9898,0.9997,0.9901,6384
5,doi seed,seed=47,0.9921,0.9920,0.9957,0.9963,6450
6,doi seed,seed=48,0.9940,0.9946,0.9994,0.9952,6419
7,doi seed,seed=49,0.9932,0.9937,0.9940,0.9997,6483
8,doi seed,seed=50,1.0000,1.0000,1.0000,1.0000,6446
9,doi seed,seed=51,0.9933,0.9938,0.9940,0.9998,6484


Doi seed : ARI 0.9775-1.0000 | Jaccard 0.9786-1.0000


#### 1.2. BOOTSTRAP STABILITY TEST: Kiểm tra mỗi rider có được phân vào Target ổn định không khi thay đổi mẫu dữ liệu và cách khởi tạo KMeans

In [40]:
# Bootstrap rider: fit lại KMeans trên mẫu bốc có hoàn lại, rồi gán TẤT CẢ rider
# bằng centroid gần nhất. Hỏi: bao nhiêu % rider giữ nguyên trạng thái target?
# Số lần bootstrap
B_BOOT = 40
rng    = np.random.default_rng(SEED)
n      = len(X)
# Đánh dấu rider nào thuộc Target gốc
# True  = thuộc Target gốc, False = không thuộc Target gốc
in_tgt = df.user_id.isin(TARGET_SET).values
# hits[i] = số lần rider i được KMeans đưa vào Target
# Ban đầu tất cả = 0
hits, jacs = np.zeros(n), []

# CHẠY BOOTSTRAP 
_t0 = time.perf_counter()
for _ in range(B_BOOT):
    # Lấy mẫu bootstrap
    # Chọn ngẫu nhiên n rider từ n rider ban đầu
    # => Một rider có thể xuất hiện nhiều lần
    # => Một số rider có thể không được chọn
    idx = rng.integers(0, n, n)

    # Chạy KMeans trên mẫu bootstrap
    # BEST_K: giữ nguyên số cluster
    # n_init=10: thử 10 cách khởi tạo centroid
    # random_state: seed ngẫu nhiên cho mỗi lần
    km  = KMeans(BEST_K, n_init=10, random_state=int(rng.integers(1_000_000))).fit(X[idx])

    # Dùng KMeans vừa học để predict TOÀN BỘ rider
    # KMeans được fit trên X[idx],
    # nhưng predict trên toàn bộ X
    # => Mỗi rider đều được kiểm tra xem có thuộc Target không
    lab = km.predict(X)
        
    # Tìm lại Target cluster
    k   = match_target_cluster(lab)

    # Đếm số lần mỗi rider được đưa vào Target
    hits += (lab == k)

    # Tính Jaccard của Target mới với Target gốc
    jacs.append(overlap(uid[lab == k])[0])

# TÍNH XÁC SUẤT MỖI RIDER THUỘC TARGET
df["p_target"] = hits / B_BOOT

# STABILITY CỦA TARGET GỐC
# Chỉ xét những rider thuộc Target gốc. Xem bao nhiêu % trong số họ vẫn được chọn làm Target ít nhất 50% số lần bootstrap
STAB_MEMBER = float((df.loc[in_tgt, "p_target"] >= .5).mean())

# Trong những rider KHÔNG thuộc Target gốc, bao nhiêu % bị KMeans đưa vào Target >= 50% số lần bootstrap
FALSE_IN    = float((df.loc[~in_tgt, "p_target"] >= .5).mean())

# Kết quả
print(f"Bootstrap {B_BOOT} lan ({time.perf_counter() - _t0:.1f}s)")
print(f"Jaccard trung binh           : {np.mean(jacs):.4f} "
      f"(min {np.min(jacs):.4f}, max {np.max(jacs):.4f})")
print(f"% rider target goc van target: {STAB_MEMBER:.1%} (o >= 1/2 so lan boot)")
print(f"% rider NGOAI bi keo vao     : {FALSE_IN:.1%}")
print(f"% rider co gan cum lung lay  : {((df.p_target > .1) & (df.p_target < .9)).mean():.1%}")

Bootstrap 40 lan (3.5s)
Jaccard trung binh           : 0.9633 (min 0.9002, max 0.9918)
% rider target goc van target: 99.8% (o >= 1/2 so lan boot)
% rider NGOAI bi keo vao     : 0.4%
% rider co gan cum lung lay  : 3.6%


## 2. Stress experiment

#### 2.1. Nếu đo bằng estimator khác, hoặc bỏ đi một khối, thì kết luận có đổi không?

Hai phép lay cùng nhắm vào một chỗ: **đặc tả mô hình** `Y ~ T + block FE`.

| Phép lay | Bắt lỗi gì | 
|---|---|
| Đổi estimator | con số báo cáo là tính chất của dữ liệu hay sản phẩm của lựa chọn mô hình | 
| Bỏ từng khối | một khối chi phối cả kết quả | 

In [ ]:
COV = ["total_rides", "recency_days", "typical_distance", "route_entropy", "pct_airport",
       "weekend_ratio", "pct_flex_payment", "pct_tip_rate", "avg_fare", "age", "is_urban"]

# Cùng một dữ liệu, ba cách tính 
_a, _b = Y_TG[T_TG == 1], Y_TG[T_TG == 0]
_dim   = _a.mean() - _b.mean()
_se_w  = np.sqrt(_a.var(ddof=1) / len(_a) + _b.var(ddof=1) / len(_b))    # sai số Welch

est = pd.DataFrame([("Hieu trung binh tho (Welch)", _dim, _se_w),
                    ("Block FE (HC1)  <- bao cao", *fe_ate_se(Y_TG, T_TG, B_TG)),
                    ("Block FE + hiep bien (HC1)",
                     *fe_ate_se(Y_TG, T_TG, B_TG, extra=tg[COV].to_numpy(float)))],
                   columns=["estimator", "ATE", "SE"])
est["KTC thap"]  = est.ATE - Z * est.SE
est["KTC cao"]   = est.ATE + Z * est.SE
est["> BREAKEVEN"] = np.where(est["KTC thap"] > BREAKEVEN, "CO", "KHONG")
display(est.round(4))

EST_SPREAD = float(est.ATE.max() - est.ATE.min())
print(f"Bien do diem uoc luong: {EST_SPREAD:.4f} chuyen = {EST_SPREAD / SE:.1f} SE")
print(f"Blocking giam SE {1 - est.SE[1] / est.SE[0]:.1%}  ->  tuong duong tang co mau "
      f"{(est.SE[0] / est.SE[1]) ** 2 - 1:.1%} ma khong ton them rider nao")
print(f"{int((est['> BREAKEVEN'] == 'CO').sum())}/{len(est)} estimator co can duoi vuot nguong hoa von.")

,estimator,ATE,SE,KTC thap,KTC cao,> BREAKEVEN
0,Hieu trung binh tho (Welch),2.0798,0.1709,1.7449,2.4146,CO
1,Block FE (HC1) <- bao cao,1.9670,0.1412,1.6902,2.2438,CO
2,Block FE + hiep bien (HC1),1.9172,0.1329,1.6566,2.1777,CO


Bien do diem uoc luong: 0.1626 chuyen = 1.2 SE
Blocking giam SE 17.3%  ->  tuong duong tang co mau 46.4% ma khong ton them rider nao
3/3 estimator co can duoi vuot nguong hoa von.


In [ ]:
# Bỏ từng khối 
# Bootstrap phân tầng và chia đôi mẫu đều bốc lại TRONG từng ô (khối × nhánh) nên mẫu
# nào cũng còn đủ 10 khối — theo thiết kế. 
_rows = []
for _bk in np.unique(B_TG):
    _in = B_TG == _bk
    _a_in,  _s_in  = fe_ate_se(Y_TG[_in],  T_TG[_in],  B_TG[_in])    # ATE RIÊNG khối đó
    _a_out, _s_out = fe_ate_se(Y_TG[~_in], T_TG[~_in], B_TG[~_in])   # ATE khi BỎ khối đó
    _rows.append({"khoi": int(_bk), "n": int(_in.sum()),
                  "total_rides TB": tg.total_rides.to_numpy(float)[_in].mean(),
                  "ATE rieng khoi": _a_in, "KTC thap rieng": _a_in - Z * _s_in,
                  "ATE khi bo khoi": _a_out, "KTC thap khi bo": _a_out - Z * _s_out})

lobo = pd.DataFrame(_rows)
lobo["> BREAKEVEN"] = np.where(lobo["KTC thap khi bo"] > BREAKEVEN, "CO", "KHONG")
display(lobo.round(4))

LOBO_SPREAD = float(lobo["ATE khi bo khoi"].max() - lobo["ATE khi bo khoi"].min())
_blk_spread = float(lobo["ATE rieng khoi"].max() - lobo["ATE rieng khoi"].min())
print(f"ATE rieng tung khoi : {lobo['ATE rieng khoi'].min():.4f} - {lobo['ATE rieng khoi'].max():.4f}"
      f"   bien do {_blk_spread:.4f} (moi khoi chi ~{int(lobo.n.mean())} rider nen KTC rat rong)")
print(f"ATE khi bo mot khoi : {lobo['ATE khi bo khoi'].min():.4f} - {lobo['ATE khi bo khoi'].max():.4f}"
      f"   bien do {LOBO_SPREAD:.4f} = {LOBO_SPREAD / SE:.1f} SE")
print(f"{int((lobo['> BREAKEVEN'] == 'CO').sum())}/{len(lobo)} lan bo khoi van co can duoi > nguong hoa von.")
# Chênh lệch giữa các khối chỉ đáng lo nếu BỎ một khối làm kết quả dịch mạnh.
print("->", "khong khoi nao chiu luc" if LOBO_SPREAD < 2 * SE else "CO khoi chi phoi ket qua")

,khoi,n,total_rides TB,ATE rieng khoi,KTC thap rieng,ATE khi bo khoi,KTC thap khi bo,> BREAKEVEN
0,0,761,2.7806,1.4826,1.1504,2.0320,1.7212,CO
1,1,651,7.6052,1.6864,1.0253,1.9985,1.6997,CO
2,2,798,11.4975,2.2679,1.5861,1.9248,1.6240,CO
3,3,662,15.0453,2.4977,1.6657,1.9062,1.6128,CO
4,4,573,17.9983,2.3522,1.4456,1.9295,1.6388,CO
5,5,680,21.4441,2.3982,1.4882,1.9161,1.6258,CO
6,6,548,25.4380,1.9051,0.8595,1.9728,1.6862,CO
7,7,602,30.3571,2.1419,1.1401,1.9490,1.6616,CO
8,8,605,37.9868,1.2020,0.1420,2.0463,1.7613,CO
9,9,566,58.2668,1.6833,0.3400,1.9944,1.7198,CO


ATE rieng tung khoi : 1.2020 - 2.4977   bien do 1.2957 (moi khoi chi ~644 rider nen KTC rat rong)
ATE khi bo mot khoi : 1.9062 - 2.0463   bien do 0.1401 = 1.0 SE
10/10 lan bo khoi van co can duoi > nguong hoa von.
-> khong khoi nao chiu luc


#### 2.2. Sai số chuẩn và p-value có đáng tin không?

Hai phép, hai câu hỏi khác nhau:

| Phép | Hỏi gì |
|---|---|
| Hoán vị Fisher | giả thuyết null có bị bác bỏ thật không | 
| Bootstrap phân tầng | khoảng tin cậy có đúng độ rộng không | 


In [ ]:
# Hoán vị Fisher 
# Mọi con số (SE, KTC, p) đều dựa vào xấp xỉ tiệm cận chuẩn. Y lệch phải
# rõ (skew ~1,1, max 52) và mỗi khối chỉ ~644 rider, nên xấp xỉ đó cần được kiểm.
# Hoán vị không xấp xỉ gì: giữ nguyên Y, chỉ tráo lại ai nhận voucher TRONG TỪNG KHỐI
# — đúng cách đã bốc thăm. Phân phối thu được là phân phối THẬT của estimator dưới
# giả thuyết "voucher không tác động lên bất kỳ ai".
B_PERM = 10_000
_rng   = np.random.default_rng(SEED)
_cells_blk = [np.flatnonzero(B_TG == b) for b in np.unique(B_TG)]

_YB    = Y_TG - pd.Series(Y_TG).groupby(B_TG).transform("mean").to_numpy()
_TB    = T_TG - pd.Series(T_TG).groupby(B_TG).transform("mean").to_numpy()
_denom = _TB @ _TB

perm = np.empty(B_PERM)
_t   = T_TG.copy()
for i in range(B_PERM):
    for ix in _cells_blk:                       # tráo nhãn T trong từng khối
        _t[ix] = _rng.permutation(T_TG[ix])
    perm[i] = (_t @ _YB) / _denom

# Kiểm đường tắt cho ra đúng con số của estimator đầy đủ
assert abs((T_TG @ _YB) / _denom - ATE) < 1e-9, "Frisch-Waugh lech so voi fe_ate_se"

p_perm = float((np.abs(perm) >= abs(ATE)).mean())          # kiểm định hai phía
print(f"Hoan vi {B_PERM:,} lan trong khoi")
print(f"  Phan phoi null : TB {perm.mean():+.4f}  SD {perm.std(ddof=1):.4f}  "
      f"(SE giai tich {SE:.4f})")
print(f"  ATE quan sat   : {ATE:.4f} — lon hon TOAN BO {(np.abs(perm) < abs(ATE)).mean():.2%} lan hoan vi")
print(f"  p hoan vi      : {'<' if p_perm == 0 else ''}{max(p_perm, 1 / B_PERM):.5f}  "
      f"(nguong {ALPHA})  ->  {'BAC BO H0' if p_perm < ALPHA else 'KHONG BAC BO'}")
# SD của phân phối null phải khớp SE giải tích — lệch nhiều nghĩa là công thức SE sai
_sd_gap = abs(perm.std(ddof=1) - SE) / SE
print(f"  SD null / SE giai tich lech {_sd_gap:.1%}  ->  "
      f"{'cong thuc SE dang tin' if _sd_gap < .10 else 'CONG THUC SE DANG NGHI'}")

Hoan vi 10,000 lan trong khoi
  Phan phoi null : TB +0.0018  SD 0.1437  (SE giai tich 0.1412)
  ATE quan sat   : 1.9670 — lon hon TOAN BO 100.00% lan hoan vi
  p hoan vi      : <0.00010  (nguong 0.05)  ->  BAC BO H0
  SD null / SE giai tich lech 1.8%  ->  cong thuc SE dang tin


In [ ]:
# Bootstrap phân tầng 
# Hoán vị kiểm GIẢ THUYẾT NULL; bootstrap kiểm ĐỘ RỘNG khoảng tin cậy. Bốc lại có hoàn lại TRONG từng ô (khối × nhánh) để mẫu mô
# phỏng giữ đúng cấu trúc thiết kế; bốc lại trên toàn mẫu sẽ phá tỷ lệ 50/50 mỗi khối
# và làm KTC rộng ra .
B_BOOT = 2_000
cells  = strata(B_TG, T_TG)                      # danh sách chỉ số của từng ô
_rng   = np.random.default_rng(SEED)

boot = np.empty(B_BOOT)
for i in range(B_BOOT):
    _ix     = np.concatenate([c[_rng.integers(0, len(c), len(c))] for c in cells])
    boot[i] = fe_ate(Y_TG[_ix], T_TG[_ix], B_TG[_ix])

LO_B, HI_B = np.percentile(boot, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
CI_GAP     = max(abs(LO_B - LO_A), abs(HI_B - HI_A))

ci = pd.DataFrame([("Giai tich (HC1)", ATE, LO_A, HI_A, HI_A - LO_A),
                   ("Bootstrap phan tang", boot.mean(), LO_B, HI_B, HI_B - LO_B)],
                  columns=["khoang tin cay", "diem", "thap", "cao", "do rong"])
display(ci.round(4))

print(f"Bootstrap {B_BOOT:,} lan tren {len(cells)} o (khoi x nhanh)")
print(f"  Lech bien lon nhat : {CI_GAP:.4f} chuyen ({CI_GAP / (HI_A - LO_A):.1%} do rong KTC)"
      f"  ->  {'KTC giai tich dang tin' if CI_GAP < .10 else 'KTC GIAI TICH DANG NGHI'}")
print(f"  Do lech bootstrap  : {boot.std(ddof=1):.4f}  (SE giai tich {SE:.4f})")
print(f"  Vuot nguong hoa von: {(boot > BREAKEVEN).mean():.1%} lan bootstrap "
      f"(nguong {BREAKEVEN:.4f})")

,khoang tin cay,diem,thap,cao,do rong
0,Giai tich (HC1),1.9670,1.6902,2.2438,0.5536
1,Bootstrap phan tang,1.9678,1.6979,2.2531,0.5552


Bootstrap 2,000 lan tren 20 o (khoi x nhanh)
  Lech bien lon nhat : 0.0093 chuyen (1.7% do rong KTC)  ->  KTC giai tich dang tin
  Do lech bootstrap  : 0.1421  (SE giai tich 0.1412)
  Vuot nguong hoa von: 100.0% lan bootstrap (nguong 1.3783)


## 3. Stress khuyến nghị

Hai mục trên lay **cách đo**. Mục này lay **giả định kinh doanh** 

| Quyết định | Điều kiện |
|---|---|
| **TRIEN KHAI** | toàn bộ KTC nằm trên ngưỡng hoà vốn — lãi  |
| **CAN NHAC** | điểm ước lượng trên ngưỡng hoà vốn nhưng KTC chạm ngưỡng |
| **DUNG** | điểm ước lượng dưới ngưỡng hoà vốn — lỗ |

In [47]:
def decide(be, ate=ATE, lo=LO_A):
    """Quy tắc quyết định ứng với một ngưỡng hoà vốn cho trước."""
    if lo  > be: return "TRIEN KHAI"
    if ate > be: return "CAN NHAC"
    return "DUNG"


fare = tg.avg_fare.mean()
# Opex là chi phí cố định mỗi rider
_be  = lambda v, r, f=None: (v + OPEX_PER_RIDER) / ((f or fare) * r)

_cost_flip = LO_A * fare * TAKE_RATE                    # chi phí tối đa chịu được
_flip = pd.DataFrame([
    ("Chi phi / rider",  f"${COST_PER_RIDER:.2f}", f"${_cost_flip:.2f}",
     _cost_flip / COST_PER_RIDER - 1),
    ("Menh gia voucher", f"${VOUCHER_VALUE:.2f}",  f"${_cost_flip - OPEX_PER_RIDER:.2f}",
     (_cost_flip - OPEX_PER_RIDER) / VOUCHER_VALUE - 1),
    ("Take rate",        f"{TAKE_RATE:.1%}",       f"{COST_PER_RIDER / (fare * LO_A):.1%}",
     COST_PER_RIDER / (fare * LO_A) / TAKE_RATE - 1),
    ("Cuoc TB / chuyen", f"${fare:.2f}",           f"${COST_PER_RIDER / (TAKE_RATE * LO_A):.2f}",
     COST_PER_RIDER / (TAKE_RATE * LO_A) / fare - 1),
], columns=["truc", "dang dung", "diem lat", "bien"])
_flip["bien"] = _flip["bien"].map(lambda x: f"{x:+.1%}")
display(_flip)
print(f"Moi dong lay mot truc, GIU NGUYEN ba truc con lai. Nguong hoa von hien tai "
      f"{BREAKEVEN:.4f} vs can duoi KTC {LO_A:.4f}.\n")

VS, RS = (3, 4, 5, 6, 8, 10, 12), (.15, .20, .25)
grid = pd.DataFrame([{"voucher ($)": v, "take rate": f"{r:.0%}", "BREAKEVEN": _be(v, r),
                      "quyet dinh": decide(_be(v, r))} for v in VS for r in RS])
display(grid.pivot(index="voucher ($)", columns="take rate", values="quyet dinh"))


OP_V, OP_R = (4, 6), (.20, .25)
_op = grid[(grid["voucher ($)"].between(*OP_V)) &
           (grid["take rate"].isin([f"{r:.0%}" for r in RS if OP_R[0] <= r <= OP_R[1]]))]
SHIP_ALL, SHIP_OP = float((grid["quyet dinh"] == "TRIEN KHAI").mean()), \
                    float((_op["quyet dinh"] == "TRIEN KHAI").mean())
print(f"Toan luoi ({len(grid)} o)      : {SHIP_ALL:.0%} TRIEN KHAI")
print(f"Vung van hanh (${OP_V[0]}-${OP_V[1]} @ {OP_R[0]:.0%}-{OP_R[1]:.0%}, {len(_op)} o): "
      f"{SHIP_OP:.0%} TRIEN KHAI")
print(f"Gia dinh dang dung ${VOUCHER_VALUE:.0f} @ {TAKE_RATE:.0%} -> {decide(BREAKEVEN)}\n")

print("Bien lat theo tung take rate:")
for r in RS:
    bad = [v for v in VS if decide(_be(v, r)) != "TRIEN KHAI"]
    print(f"  take rate {r:.0%}: " +
          (f"lat tu voucher ${min(bad)} tro len" if bad else "khong muc voucher nao lat"))

,truc,dang dung,diem lat,bien
0,Chi phi / rider,$6.25,$7.66,+22.6%
1,Menh gia voucher,$5.00,$6.41,+28.3%
2,Take rate,20.0%,16.3%,-18.5%
3,Cuoc TB / chuyen,$22.67,$18.49,-18.5%


Moi dong lay mot truc, GIU NGUYEN ba truc con lai. Nguong hoa von hien tai 1.3783 vs can duoi KTC 1.6902.



take rate,15%,20%,25%
voucher ($),,,
3,TRIEN KHAI,TRIEN KHAI,TRIEN KHAI
4,TRIEN KHAI,TRIEN KHAI,TRIEN KHAI
5,CAN NHAC,TRIEN KHAI,TRIEN KHAI
6,DUNG,TRIEN KHAI,TRIEN KHAI
8,DUNG,DUNG,TRIEN KHAI
10,DUNG,DUNG,DUNG
12,DUNG,DUNG,DUNG


Toan luoi (21 o)      : 52% TRIEN KHAI
Vung van hanh ($4-$6 @ 20%-25%, 6 o): 100% TRIEN KHAI
Gia dinh dang dung $5 @ 20% -> TRIEN KHAI

Bien lat theo tung take rate:
  take rate 15%: lat tu voucher $5 tro len
  take rate 20%: lat tu voucher $8 tro len
  take rate 25%: lat tu voucher $10 tro len
